In [1]:
!pip install -q torch_geometric scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
from google.colab import files

uploaded = files.upload()


Saving Labelled Yelp Dataset.csv to Labelled Yelp Dataset.csv


In [2]:
DATA_PATH = "Labelled Yelp Dataset.csv"

In [3]:
import pandas as pd

df = pd.read_csv(
    DATA_PATH,
    engine="python",
    on_bad_lines="skip"
)

print("Raw shape:", df.shape)
print(df.head())
print(df.columns)


Raw shape: (39524, 6)
   User_id  Product_id  Rating       Date  \
0      923           0       3  12/8/2014   
1      924           0       3  5/16/2013   
2      925           0       4   7/1/2013   
3      926           0       4  7/28/2011   
4      927           0       4  11/1/2010   

                                              Review  Label  
0  The food at snack is a selection of popular Gr...     -1  
1  This little place in Soho is wonderful. I had ...     -1  
2  ordered lunch for 15 from Snack last Friday. Â...     -1  
3  This is a beautiful quaint little restaurant o...     -1  
4  Snack is great place for a Â casual sit down l...     -1  
Index(['User_id', 'Product_id', 'Rating', 'Date', 'Review', 'Label'], dtype='object')


In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix

import torch
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


DATA_PATH = "Labelled Yelp Dataset.csv"


TEXT_COL  = "Review"
LABEL_COL = "Label"


K_NEIGHBORS  = 10
MAX_FEATURES = 5000
VAL_SIZE     = 0.1
TEST_SIZE    = 0.2
RANDOM_STATE = 42

N_SAMPLE     = 50000


Using device: cuda


In [5]:
df = pd.read_csv(
    DATA_PATH,
    engine="python",
    on_bad_lines="skip"
)

print("Raw shape:", df.shape)
print(df.head())
print(df.columns)


Raw shape: (77317, 6)
   User_id  Product_id  Rating       Date  \
0      923           0       3  12/8/2014   
1      924           0       3  5/16/2013   
2      925           0       4   7/1/2013   
3      926           0       4  7/28/2011   
4      927           0       4  11/1/2010   

                                              Review  Label  
0  The food at snack is a selection of popular Gr...     -1  
1  This little place in Soho is wonderful. I had ...     -1  
2  ordered lunch for 15 from Snack last Friday. Â...     -1  
3  This is a beautiful quaint little restaurant o...     -1  
4  Snack is great place for a Â casual sit down l...     -1  
Index(['User_id', 'Product_id', 'Rating', 'Date', 'Review', 'Label'], dtype='object')


In [6]:
df = pd.read_csv(DATA_PATH)
df.head()

ParserError: Error tokenizing data. C error: EOF inside string starting at row 83590

In [7]:
df = df[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
print("After dropping NA:", df.shape)

df[LABEL_COL] = df[LABEL_COL].astype(int).map({-1: 0, 1: 1})

print("Label distribution before subsample (0=fake, 1=genuine):")
print(df[LABEL_COL].value_counts())

if len(df) > N_SAMPLE:
    df = df.sample(n=N_SAMPLE, random_state=RANDOM_STATE).reset_index(drop=True)
    print("\nUsing subset of size:", len(df))
else:
    print("\nUsing full data of size:", len(df))

print("Label distribution after subsample:")
print(df[LABEL_COL].value_counts())
df.head()


After dropping NA: (77317, 2)
Label distribution before subsample (0=fake, 1=genuine):
Label
1    69337
0     7980
Name: count, dtype: int64

Using subset of size: 50000
Label distribution after subsample:
Label
1    44812
0     5188
Name: count, dtype: int64


,Review,Label
0,Saw this place on TV (food network or travel c...,1
1,A place to see and be seen. I am baffled when ...,1
2,For a place that specializes in Chicken and Wa...,1
3,":-( Â + Â ||| Â = Â :)P , then :D I really lov...",1
4,So I went to Cookshop a few years ago when the...,1


In [8]:
indices = np.arange(len(df))
labels  = df[LABEL_COL].values

train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels,
)

train_labels = labels[train_idx]
train_idx, val_idx = train_test_split(
    train_idx,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=RANDOM_STATE,
    stratify=train_labels,
)

print("Train size:", len(train_idx))
print("Val size:  ", len(val_idx))
print("Test size: ", len(test_idx))


Train size: 35000
Val size:   5000
Test size:  10000


In [9]:
vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=(1, 2),
    stop_words="english",
)

X = vectorizer.fit_transform(df[TEXT_COL].astype(str).values)
print("TF-IDF shape:", X.shape)

X_dense = torch.tensor(X.toarray(), dtype=torch.float)
y = torch.tensor(df[LABEL_COL].values, dtype=torch.long)

num_nodes, num_features = X_dense.shape
print("Num nodes:", num_nodes, "Num features:", num_features)


TF-IDF shape: (50000, 5000)
Num nodes: 50000 Num features: 5000


In [10]:
nn_model = NearestNeighbors(
    n_neighbors=K_NEIGHBORS + 1,
    metric="cosine",
)
nn_model.fit(X)

distances, indices_knn = nn_model.kneighbors(X)
print("kNN indices shape:", indices_knn.shape)

edge_index_list = []
for i in range(num_nodes):
    neighbors = indices_knn[i, 1:]
    for j in neighbors:
        edge_index_list.append([i, j])
        edge_index_list.append([j, i])

edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
print("edge_index shape:", edge_index.shape)


kNN indices shape: (50000, 11)
edge_index shape: torch.Size([2, 1000000])


In [11]:
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask   = torch.zeros(num_nodes, dtype=torch.bool)
test_mask  = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx]     = True
test_mask[test_idx]   = True

print("Train nodes:", train_mask.sum().item())
print("Val nodes:",   val_mask.sum().item())
print("Test nodes:",  test_mask.sum().item())


Train nodes: 35000
Val nodes: 5000
Test nodes: 10000


In [12]:
data = Data(
    x=X_dense,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask,
)

data = data.to(device)
data


Data(x=[50000, 5000], edge_index=[2, 1000000], y=[50000], train_mask=[50000], val_mask=[50000], test_mask=[50000])

In [13]:
class SAGEBackbone(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

class ReviewGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.backbone = SAGEBackbone(in_channels, hidden_channels, num_classes)

    def forward(self, x, edge_index):
        return self.backbone(x, edge_index)

in_channels = data.x.size(1)
hidden_size = 64
num_classes = len(torch.unique(data.y))

model = ReviewGNN(in_channels, hidden_size, num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

print(model)


ReviewGNN(
  (backbone): SAGEBackbone(
    (conv1): SAGEConv(5000, 64, aggr=mean)
    (conv2): SAGEConv(64, 2, aggr=mean)
    (relu): ReLU()
    (dropout): Dropout(p=0.5, inplace=False)
  )
)


In [14]:
def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    preds = out[data.train_mask].argmax(dim=1)
    labels = data.y[data.train_mask]
    acc = (preds == labels).float().mean().item()
    return loss.item(), acc


@torch.no_grad()
def evaluate(mask):
    model.eval()
    out = model(data.x, data.edge_index)
    preds = out[mask].argmax(dim=1)
    labels = data.y[mask]
    acc = (preds == labels).float().mean().item()
    return acc, preds.cpu().numpy(), labels.cpu().numpy()


In [15]:
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch()
    val_acc, _, _ = evaluate(data.val_mask)

    if epoch == 1 or epoch % 2 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )


Epoch 001 | Train Loss: 0.6300 | Train Acc: 0.8963 | Val Acc: 0.8962
Epoch 002 | Train Loss: 0.6151 | Train Acc: 0.8963 | Val Acc: 0.8962
Epoch 004 | Train Loss: 0.5873 | Train Acc: 0.8963 | Val Acc: 0.8962
Epoch 006 | Train Loss: 0.5570 | Train Acc: 0.8963 | Val Acc: 0.8962
Epoch 008 | Train Loss: 0.5261 | Train Acc: 0.8963 | Val Acc: 0.8962
Epoch 010 | Train Loss: 0.4948 | Train Acc: 0.8963 | Val Acc: 0.8962


In [24]:
test_acc, y_pred, y_true = evaluate(data.test_mask)
print(f"\nTest Accuracy: {test_acc:.4f}")

print("\nClassification report (test):")
print(classification_report(y_true, y_pred))

print("\nConfusion matrix (test):")
print(confusion_matrix(y_true, y_pred))



Test Accuracy: 0.8948

Classification report (test):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1052
           1       0.89      1.00      0.94      8948

    accuracy                           0.89     10000
   macro avg       0.45      0.50      0.47     10000
weighted avg       0.80      0.89      0.85     10000


Confusion matrix (test):
[[   0 1052]
 [   0 8948]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
